In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import pandas as pd
from matplotlib.lines import Line2D

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.964, 0.912, 0.849, 0.691, 0.314]
B_eff = [0.013, 0.013, 0.013, 0.013, 0.013]
B_cs = 87
L1 = 25 #ab-1，1 ab⁻¹ = 1000 fb⁻¹
L2 = 150 #ab-1
# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP['xs_mumu_prompt'] = pho_ALP['xs_mumu_prompt'].astype('float32')
pho_ALP['Lambda'] = 1000 / pho_ALP['Lambda']

xs = pho_ALP['xs_mumu_prompt']
gstar = pho_ALP['Lambda']
MQ = pho_ALP['ma']

In [ ]:
values_os

In [ ]:
griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os))

In [ ]:
griddensity = 400
# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), griddensity)
gstar_14_os = np.linspace(gstar.min(), gstar.max(), griddensity)

# 计算246/gstar的y轴
y_axis = gstar_14_os

# 将y_axis转换为与MQ_grid_14_os相同的形状
y_axis_grid = np.tile(y_axis, (mq_14_os.shape[0], 1))  # 广播到2D

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 使用二维插值来计算网格上的 S 和 Z
points_os = pho_ALP[['ma', 'Lambda']].values
values_os = pho_ALP['xs_mumu_prompt'].values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)

# 重新计算S_grid和背景
S_grid_14_os_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  # 3fb
background_14_os_emu = B_cs * L1 * 1000 * 0.013 * 1000  # 3fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_emu = S_grid_14_os_emu / np.sqrt(S_grid_14_os_emu + background_14_os_emu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_emu = S_grid_14_os_30_emu / np.sqrt(S_grid_14_os_30_emu + background_14_os_30_emu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3emup = Z_grid_14_os_emu+np.sqrt(std_3_emu)
z3emum = Z_grid_14_os_emu-np.sqrt(std_3_emu)

std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30emup = Z_grid_14_os_30_emu+np.sqrt(std_30_emu)
z30emum = Z_grid_14_os_30_emu-np.sqrt(std_30_emu)

In [ ]:
# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_emu, levels=[2], colors='blue', label='Z_grid_14_os_30')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_emu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emup, levels=[2], colors='blue', label='Z_grid_14_os_30p',linestyles='--')
contour_emu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emum, levels=[2], colors='blue', label='Z_grid_14_os_30m',linestyles='--')
upper_segments_30 = contour_emu30p.allsegs[0]
lower_segments_30 = contour_emu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='lightblue', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_emu, levels=[2], colors='red', label='Z_grid_14_os')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_emu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emup, levels=[2], colors='red', label='Z_grid_14_os_30p',linestyles='--')
contour_emu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emum, levels=[2], colors='red', label='Z_grid_14_os_30m',linestyles='--')
upper_segments_3 = contour_emu3p.allsegs[0]
lower_segments_3 = contour_emu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='lightpink', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
#plt.ylim(1e-2,1e0)
plt.xscale('log')
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.yscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='r', lw=2, label=R'2$\sigma$ C.L. at 25 ab$^{-1}$'),
    Line2D([0], [0], color='b', lw=2, label=R'2$\sigma$ C.L. at 150 ab$^{-1}$ '),
    Line2D([0], [0], color='r', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ boundary'),
    Line2D([0], [0], color='b', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ boundary')
]
plt.text(0.3, 0.8, R'$a \to \mu^{+} \mu^{-}$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=20, color='black')
plt.legend(handles=legend_elements, loc='lower right', fontsize=15)
plt.savefig('./amumu.png',dpi=300)
plt.show()